# Experimento Completo no Colab

Este é o notebook principal do projeto. Ele foi preparado para executar o experimento completo em um ambiente Jupyter ou Google Colab, reutilizando os pipelines já implementados no repositório.

## O que este notebook faz

1. prepara o repositório no ambiente;
2. instala as dependências necessárias;
3. localiza o dataset POP909;
4. gera um arquivo `config/colab.yaml` sem alterar a configuração padrão do projeto;
5. executa o pipeline completo com `python src/main.py all`;
6. mostra onde os principais resultados foram gerados.

## Pré-requisitos

- o dataset POP909 precisa estar disponível;
- no Colab, o caminho mais comum é via Google Drive;
- se o dataset já estiver em `data/raw/POP909`, basta seguir normalmente.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/stef325/plagiarismDetectMethodsEval.git"
PROJECT_DIR = Path("/content/plagiarismDetectMethodsEval")
IN_COLAB = "google.colab" in sys.modules

def find_existing_project() -> Path | None:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src" / "main.py").exists():
            return candidate
    return None

existing_project = find_existing_project()

if existing_project is not None:
    PROJECT_DIR = existing_project
elif IN_COLAB:
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    raise FileNotFoundError(
        "Nao foi possivel localizar o repositorio atual. Abra este notebook dentro do projeto ou execute no Colab."
    )

os.chdir(PROJECT_DIR)
print(f"Projeto em uso: {PROJECT_DIR}")


In [ ]:
USE_GOOGLE_DRIVE = True
DATASET_SOURCE = Path("/content/drive/MyDrive/POP909")

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

default_dataset_path = PROJECT_DIR / "data" / "raw" / "POP909"

if default_dataset_path.exists():
    DATASET_PATH = default_dataset_path
elif DATASET_SOURCE.exists():
    DATASET_PATH = DATASET_SOURCE
else:
    raise FileNotFoundError(
        "Dataset POP909 nao encontrado. Coloque o dataset em data/raw/POP909 ou ajuste DATASET_SOURCE para o caminho correto."
    )

print(f"Dataset em uso: {DATASET_PATH}")


In [ ]:
%pip install -r requirements.txt

In [ ]:
import yaml

default_config_path = PROJECT_DIR / "config" / "default.yaml"
colab_config_path = PROJECT_DIR / "config" / "colab.yaml"

with default_config_path.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

config["dataset"]["path"] = str(DATASET_PATH)

with colab_config_path.open("w", encoding="utf-8") as file:
    yaml.safe_dump(config, file, sort_keys=False, allow_unicode=True)

print(f"Configuracao gerada em: {colab_config_path}")


In [ ]:
import time

def run_command(command: list[str]) -> None:
    print("Executando:", " ".join(command))
    start = time.time()
    subprocess.run(command, check=True)
    elapsed = time.time() - start
    print(f"Tempo da etapa: {elapsed:.2f} segundos")


## Execução completa

A célula abaixo executa o fluxo completo do experimento usando o arquivo `config/colab.yaml` gerado especificamente para este ambiente.

In [ ]:
run_command([
    sys.executable,
    "src/main.py",
    "--config",
    "config/colab.yaml",
    "all",
])


In [ ]:
from IPython.display import Markdown, display

important_outputs = [
    PROJECT_DIR / "data" / "results" / "experiment" / "similarity_results.csv",
    PROJECT_DIR / "data" / "results" / "evaluation" / "robustness_metrics.csv",
    PROJECT_DIR / "data" / "results" / "evaluation" / "interpretability" / "interpretability_results.csv",
    PROJECT_DIR / "data" / "results" / "consolidated" / "consolidated_results.md",
    PROJECT_DIR / "data" / "results" / "figures" / "visualizations.md",
]

for output in important_outputs:
    print(f"{output.exists()} -> {output}")

consolidated_report = PROJECT_DIR / "data" / "results" / "consolidated" / "consolidated_results.md"
if consolidated_report.exists():
    display(Markdown(consolidated_report.read_text(encoding="utf-8")))
else:
    print("Relatorio consolidado ainda nao foi encontrado.")


## Próximos notebooks úteis

- `01_step_by_step_execution.ipynb`: execução controlada por etapas;
- `02_results_analysis.ipynb`: leitura rápida dos resultados já gerados.
